In [ ]:
import os

# set current directory to where the 'CBH.py' is located or Move this file in the same directory
module_dir = r"D:/ForestFire/CrownFire/src"
os.chdir(module_dir)

from CBH import nfiPreprocessing, func, loss_func, species_weighted_r2, objective, printFeatureImportance, \
hybrid_model, create_valid_mask, buildDistribution, visualize_multiple_distribution, GPUSamplingImsang, \
check_model_validity, write_log, rasterize_feature, quick_check, plot_raster

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, traceback
from glob import glob
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split, StratifiedKFold
from scipy.optimize import minimize
import joblib
from tqdm.notebook import tqdm
import matplotlib.font_manager as fm
import numpy as np
import random
from datetime import datetime

# Libraries for ML-based learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import xgboost as xgb
from xgboost import XGBRegressor
import optuna
from sklearn.model_selection import cross_val_score, RandomizedSearchCV, GridSearchCV, KFold, GroupKFold
from sklearn.metrics import make_scorer
from sklearn.ensemble import RandomForestRegressor
import shap
from xgboost import plot_importance
import warnings
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor
from functools import partial
import joblib

# Libraries for the pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin

# Libraries for the spatial analysis 
import geopandas as gpd
import scipy.stats as stats
import rasterio
import fiona
from rasterio.windows import Window
from collections import defaultdict
from rasterio.transform import xy
from rasterio.windows import transform
from rasterio.features import rasterize
from rasterio.plot import show

try:
    import pyproj
    from pyproj import CRS
except ImportError as e:
    print(e)
    usr_site = site.getusersitepackages()
    if usr_site in sys.path:
        sys.path.remove(usr_site)   # stop picking up pip user packages
        print("Removed user site:", usr_site)
    
    # Now import safely
    import pyproj
    from pyproj import CRS
    print("pyproj OK") 
    
# Libraries for the GPU use
import cupy as cp

# warning filter
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

In [ ]:
# ==== 경로 설정 ====
parent_dir = r"D:\ForestFire\CBH\test"
data_dir = os.path.join(parent_dir, "data")
result_dir = os.path.join(parent_dir, "result")
model_dir = os.path.join(parent_dir, r"result\model")
fig_dir = os.path.join(parent_dir, r"result\fig")
dir_list = [data_dir, result_dir, model_dir, fig_dir]

for dir_name in dir_list:
    if not os.path.exists(dir_name):
        os.mkdir(dir_name)

raster_dir = r"F:\CBH"
raster_result_dir = r"F:\CBH\test"
gdb_dir = r"H:\CBH\Imsang_merge.gdb"

# Global random seed (random operations: numpy, pandas, StratifiedKfold, train_test_split)
SEED = 200 # 100
random.seed(SEED)
np.random.seed(SEED)
try_num = "HM변형-try1.0"

# 단위 변환
cm_to_inch = 0.393701
m_to_ft = 3.28084

# If you train the model..
pipeline_name = f'{try_num}-XGBoostGlobal-pipeline.pkl'

In [ ]:
# ==== Map names to codes ====
name_lst1 =  ['소나무', '잣나무', '낙엽송', '리기다소나무', '곰솔', '전나무', '편백나무', '삼나무', '가문비나무', '비자나무', '은행나무', '상수리나무', '신갈나무', 
             '굴참나무', '기타참나무류', '오리나무', '고로쇠나무', '자작나무', '박달나무', '밤나무', '물푸레나무', '서어나무', '때죽나무', '호두나무', '백합나무', 
             '포플러', '벚나무', '느티나무', '층층나무', '아까시나무', '가시나무', '구실잣밤나무', '녹나무', '굴거리나무', '황칠나무','사스레피나무', '후박나무','새덕이']
name_lst2 = ['기타침엽수', '기타참나무류', '기타활엽수']  # same as your input
id_lst = [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 61, 62, 63, 64, 65, 66, 67, 68]     # same as your input
code_name_dict = {j: i for i, j in zip(id_lst, name_lst1)}
code_name_dict.update({'기타침엽수': 10, '기타활엽수': 30, '기타참나무류': 34})
reversed_dict = {value : key for key, value in code_name_dict.items()}

# ==== NFI 전처리 ====
nfi_cleaned = r"NFI-Integrated(6-7)-cleaned.csv"
nfi_file = os.path.join(data_dir, nfi_cleaned)
try:
    df = pd.read_excel(nfi_file, sheet_name = "NFI6-7")
except:
    print("Sheet name 'NFI6-7' need to exist in the NFI file.")
df_clean = nfiPreprocessing(df, nfi_file)
df_all = df_clean.reset_index()

# Model Training

In [ ]:
# ==== Allometric Model Training ====
# Global random seed (random operations: numpy, pandas, StratifiedKfold, train_test_split)
SEED = 100 # default: 100
random.seed(SEED)
np.random.seed(SEED)

# Configuration
lam = 0
test_ratio = 0.3 # default: 0.15
n_fold = 5
params_num = 4
cols = ['DBH(inch)', 'H(ft)', 'CR', 'Cycle']
os.makedirs(result_dir, exist_ok=True)
target_trees = np.unique(df_all.SID) # [code_name_dict[i] for i in valid_species]
rec = np.zeros((len(target_trees), 17 + params_num), dtype=object)
rec[:, 0] = target_trees

# Collector for unified test set
total_test_list = []
total_train_list = []

for i, sid in enumerate(tqdm(target_trees), 1):
    print(f"[{i}/{len(target_trees)}] Processing SID: {sid}")
    condition = (df_all['SID'] == sid)
    nfi6 = df_all.query("Cycle == 6").loc[condition]
    nfi7 = df_all.query("Cycle == 7").loc[condition]

    cnt = (len(nfi6), len(nfi7))
    print(cnt)
    if nfi6.isnull().values.any() or nfi7.isnull().values.any():
        print("Null data detected, skipping SID:", sid)
        continue

    # Split train/test
    if len(nfi6) >= 180:
        nfi6_test = nfi6.sample(frac=test_ratio, random_state=SEED)
        nfi7_test = nfi7.sample(frac=test_ratio, random_state=SEED)
        df_train = pd.concat([nfi6.drop(nfi6_test.index).assign(Cycle=6),nfi7.drop(nfi7_test.index).assign(Cycle=7)])

    elif (len(nfi6) < 180) & (len(nfi7) >= 30):
        # nfi6_test = nfi6
        nfi7_test = nfi7.sample(frac=test_ratio, random_state=SEED)
        df_train = nfi7.drop(nfi7_test.index).assign(Cycle=7)

    else:
        print(f"Skip {sid}: Not enough number of the samples")
        best_params = [np.nan] * params_num
        rec[rec[:, 0] == sid, :] = [
        sid, cnt, np.nan, np.nan, np.nan,
        np.nan, np.nan, np.nan,
        np.nan, np.nan, np.nan,
        np.nan, np.nan, np.nan,
        np.nan, np.nan, np.nan
        ] + list(best_params)
        continue

    # X, y array 만들기
    X = df_train[cols].drop(columns=['CR'])
    y = df_train['CR']
    stratify_col = df_train['Cycle']

    # score 변수 initialization
    best_score, worst_score, best_params = -np.inf, np.inf, None
    cv_scores = []

    print("lenght of training dataset: ", len(X))
    if len(X) >= 30:
        kf = StratifiedKFold(n_splits=n_fold, shuffle=True, random_state=SEED)
        for train_idx, val_idx in kf.split(X, stratify_col):
            X_train = X.iloc[train_idx].drop(columns='Cycle').values.T
            y_train = y.iloc[train_idx].values
            popt, _ = curve_fit(func, X_train, y_train, maxfev=10000)
            result_reg = minimize(loss_func, x0=popt, args=(lam, X_train, y_train))
            opt_params = result_reg.x
            score = r2_score(y_train, func(X_train, *opt_params))
            cv_scores.append(score)
            if score > best_score:
                best_score = score
                best_params = opt_params
            if score < worst_score:
                worst_score = score
    else: continue

    # Evaluate on full training data
    X_full = X.drop(columns='Cycle').values.T
    y_full = y
    y_pred_full = func(X_full, *best_params)
    r2_train = r2_score(y_full, y_pred_full)
    mae_train = mean_absolute_error(y_full, y_pred_full)
    rmse_train = root_mean_squared_error(y_full, y_pred_full)

    # Evaluate on test sets
    X6_test = nfi6_test[cols].drop(columns=['CR', 'Cycle']).values.T
    y6_test = nfi6_test['CR']
    X7_test = nfi7_test[cols].drop(columns=['CR', 'Cycle']).values.T
    y7_test = nfi7_test['CR']
        
    # test dataset 예측 (NFI6, NFI7, NFI6+7)
    y_pred6 = func(X6_test, *best_params)
    y_pred7 = func(X7_test, *best_params)
    
    # test dataset score 산출
    r2_test_nfi6 = r2_score(y6_test, y_pred6)
    r2_test_nfi7 = r2_score(y7_test, y_pred7)
    mae_test_nfi6 = mean_absolute_error(y6_test, y_pred6)
    mae_test_nfi7 = mean_absolute_error(y7_test, y_pred7)
    rmse_test_nfi6 = root_mean_squared_error(y6_test, y_pred6)
    rmse_test_nfi7 = root_mean_squared_error(y7_test, y_pred7)
    y_all_true = y7_test
    y_all_pred = y_pred7
    r2_test_all = r2_score(y_all_true, y_all_pred)
    mae_test_all = mean_absolute_error(y_all_true, y_all_pred)
    rmse_test_all = root_mean_squared_error(y_all_true, y_all_pred)
   
    # test-train dataset list에 저장
    df_train['CR_pred'] = np.clip(y_pred_full, 0, 1)
    total_train_list.extend([df_train])
    nfi6_test['CR_pred'] = np.clip(y_pred6, 0, 1)
    nfi7_test['CR_pred'] = np.clip(y_pred7, 0, 1)
    total_test_list.extend([nfi6_test, nfi7_test])

    # record list에 score 결과 저장
    rec[rec[:, 0] == sid, :] = [
        sid, cnt, np.mean(cv_scores), best_score, worst_score,
        r2_train, mae_train, rmse_train,
        r2_test_all, mae_test_all, rmse_test_all,
        r2_test_nfi6, mae_test_nfi6, rmse_test_nfi6,
        r2_test_nfi7, mae_test_nfi7, rmse_test_nfi7
    ] + list(best_params)

# Save results into a textfile
header = ['SID', 'Count', 'CV_Mean', 'CV_Best', 'CV_Worst',
        'R2_Train', 'MAE_Train', 'RMSE_Train',
        'R2_Test_All', 'MAE_Test_All', 'RMSE_Test_All',
        'R2_Test_NFI6', 'MAE_Test_NFI6', 'RMSE_Test_NFI6',
        'R2_Test_NFI7', 'MAE_Test_NFI7', 'RMSE_Test_NFI7']
np.savetxt(
    os.path.join(result_dir, f'Evaluation_{try_num}.txt'),
    rec, delimiter=',', fmt='%s',
    header=','.join(header + [f'Coef{i+1}' for i in range(11)]),
    comments=''
)

# Save unified test dataset
if total_test_list:
    pd.concat(total_test_list).to_csv(os.path.join(result_dir, f"NFI6+7_test_combined_{try_num}.csv"), index=False, encoding='cp949')
else:
    print("Warning: No test data collected, skipping test dataset save.")

# Save unified train dataset
if total_train_list:
    pd.concat(total_train_list).to_csv(os.path.join(result_dir, f"NFI6+7_train_combined_{try_num}.csv"), index=False, encoding='cp949')
else:
    print("Warning: No test data collected, skipping test dataset save.")


# ==== Model Validation & Evaludation ====
df_result = pd.DataFrame(rec, columns=header + [f'Par{i}' for i in range(params_num)])
s_names = [reversed_dict[i] for i in df_result['SID']]
df_result.insert(1, 'SName', s_names)
r2_columns = ['CV_Mean', 'CV_Best', 'CV_Worst', 'R2_Train', 'R2_Test_All', 'R2_Test_NFI6', 'R2_Test_NFI7']
df_result.loc[:, r2_columns] = df_result.loc[:, r2_columns].clip(lower=0) # .applymap(lambda x: 0 if x < 0 else x)
df_result.to_csv(os.path.join(result_dir, f'Evaluation_{try_num}.csv'), encoding='cp949')

In [ ]:
# ==== Print Accuracy ====
# Set font for Korean
plt.rc('font', family='Malgun Gothic')  
plt.rcParams['axes.unicode_minus'] = False  

# Read data
title = f"Evaluation_{try_num}"
df_result = pd.read_csv(os.path.join(result_dir, title + '.csv'), encoding='cp949')

# Set figure
plt.figure(figsize=(20, 6))
bar_width = 0.15
x = np.arange(len(df_result))

# Plot bars side-by-side
plt.bar(x - 2*bar_width, df_result["CV_Mean"], width=bar_width, label="CV_Mean", color="blue", alpha=0.7)
plt.bar(x - bar_width, df_result["R2_Train"], width=bar_width, label="R2_Train", color="orange", alpha=0.7)
plt.bar(x, df_result["R2_Test_All"], width=bar_width, label="R2_Test_All", color="purple", alpha=0.7)
plt.bar(x + bar_width, df_result["R2_Test_NFI6"], width=bar_width, label="R2_Test_NFI6", color="pink", alpha=0.7)
plt.bar(x + 2*bar_width, df_result["R2_Test_NFI7"], width=bar_width, label="R2_Test_NFI7", color="skyblue", alpha=0.7)

# Annotate values above bars
for i in range(len(df_result)):
    plt.text(x[i] - 2*bar_width, df_result["CV_Mean"][i] + 0.01, f'{df_result["CV_Mean"][i]:.2f}', ha='center', fontsize=6)
    plt.text(x[i] - bar_width, df_result["R2_Train"][i] + 0.01, f'{df_result["R2_Train"][i]:.2f}', ha='center', fontsize=6)
    plt.text(x[i], df_result["R2_Test_All"][i] + 0.01, f'{df_result["R2_Test_All"][i]:.2f}', ha='center', fontsize=6)
    plt.text(x[i] + bar_width, df_result["R2_Test_NFI6"][i] + 0.01, f'{df_result["R2_Test_NFI6"][i]:.2f}', ha='center', fontsize=6)
    plt.text(x[i] + 2*bar_width, df_result["R2_Test_NFI7"][i] + 0.01, f'{df_result["R2_Test_NFI7"][i]:.2f}', ha='center', fontsize=6)

# X labels and layout
x_labels = [f"{df_result.loc[i, 'SName']}{df_result.loc[i, 'Count']}" for i in range(len(df_result))]
plt.xticks(ticks=x, labels=x_labels, rotation=45, ha="right")
plt.xlabel('SName', fontsize=12)
plt.ylabel("R2 Value", fontsize=12)
plt.title(title, fontsize=14)
plt.legend()
plt.tight_layout()

# Save and show
plt.savefig(os.path.join(fig_dir, f'{title}.png'))
plt.show()

In [ ]:
# check the xgboostregressor version: SHOULD be 2.1.4
print(xgb.__version__)
print(XGBRegressor)

In [ ]:
# ==== Allometric model 예측 결과 읽어오기 ====
data_file = f"NFI6+7_train_combined_{try_num}.csv"
df = pd.read_csv(os.path.join(result_dir, data_file), encoding='cp949')

# ==== XGBoost Training using Optuna ====
# Prepare data
le = LabelEncoder()
df['SID_ENC'] = le.fit_transform(df['SID'])

feature_cols = ['H(ft)', 'DBH(inch)', 'CD(%)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'SID_ENC', 'Lat', 'Long', 'CR_pred']
cat_col = 'SID'
target_col = 'CR'

X = df[feature_cols]
y = df[target_col]
species = df[cat_col].astype(str)

# Run Optuna
optuna_objective = partial(objective, kf_split=5, SEED=SEED, X=X, y=y, groups=species)
n_trials = 10

study = optuna.create_study(direction="minimize")
study.optimize(optuna_objective, n_trials=n_trials, show_progress_bar=True)


In [ ]:
print("Best parameters:", study.best_params)
print("Best score:", study.best_value)

# iteration별 정확도 산축
# df_trials = study.trials_dataframe(attrs=("number", "value", "params", "user_attrs"))
# df_trials[["number", "user_attrs_weighted_r2"]]

# Evaluation: test data
# predict with test data
best_params = study.best_params
best_params['n_estimators'] = 500
best_params['eval_metric'] = "rmse"

# read test file
test_file = f"NFI6+7_test_combined_{try_num}.csv"
df_test = pd.read_csv(os.path.join(result_dir, test_file), encoding='cp949')

# label encoding
df_test['SID_ENC'] = le.transform(df_test['SID'])

# X, Y
X_test = df_test[feature_cols]
y_test = df_test[target_col]
groups_test = df_test['SID']

# build model with best params
model = XGBRegressor(**best_params)
model.fit(X, y)
print("Best parameters: ", model.get_params)

# predict with test dataset
y_pred_test = model.predict(X_test)

total_r2 = r2_score(y_test, y_pred_test)
print("R² on test dataset: ", total_r2)


# ==== 수종별 정확도 계산 및 저장하기 ====
groups_test = df_test['SID']
df_pred = pd.DataFrame({'true': y_test, 'pred': y_pred_test, 'group': groups_test})
group_cnt_lst = df_pred['group'].value_counts()
scores = []
rmses = []
maes = []

for sid, group_df in df_pred.groupby('group'):
    if len(group_df) >= 2:
        r2 = r2_score(group_df['true'], group_df['pred'])
        rmse = root_mean_squared_error(group_df['true'], group_df['pred'])
        mae = mean_absolute_error(group_df['true'], group_df['pred'])
        scores.append(r2)
        rmses.append(rmse)
        maes.append(mae)

df_score= pd.DataFrame({"SID" : df_test['SID'].unique(), "I_Species" : df_test['I_Species'].unique(), "R2" : scores, "RMSE" : rmses, "MAE" : maes})
accuracy_file_name = f"XGBoost_{try_num}_accuracy.csv"
print("Save the score by species...")
df_score.to_csv(os.path.join(result_dir, accuracy_file_name), encoding='cp949')
# ==== Save the result into the DataFrame ==== 
print("Save the final results...")
xg_pred = model.predict(X)
df['XG_pred'] = xg_pred
df['XG_residual'] = df['CR'] - df['XG_pred']
df.to_csv(os.path.join(result_dir, f"Hybrid-Results_{try_num}.csv"), encoding='cp949')

In [ ]:
# ==== Visualize: feature importance ====
printFeatureImportance(X_test, model)

In [ ]:
# ==== Visualize: SHAP value ====
X = df[feature_cols]
explainer = shap.Explainer(model)
shap_values = explainer(X)

plt.rcParams['font.size'] = 15 
fig, ax = plt.subplots(figsize=(15, 8))
shap.summary_plot(shap_values, X, show=False, plot_size=(15, 8))

ax = plt.gca()
fig = plt.gcf()

# Add vertical gridlines (behind points)
ax.grid(axis='x', linestyle='-', color='gray', alpha=0.4)
ax.set_axisbelow(True)

# Replace y-tick labels with more human-friendly names
pretty_names = {
    'CR_pred': f'Allometric\nPrediction',
    'Lat': 'Latitude',
    'SID_ENC': 'Species',
    'Elev(hm)': 'Elevation (hm)',
    'Long': 'Longitude',
    'H(ft)': 'Height (ft)',
    'CD(%)': 'Crown Density (%)',
    'Slope(tan)': 'Slope (tan)',
    'DBH(inch)': 'DBH (inch)',
    'Azimuth(rad)': 'Azimuth (rad)'
}

yticklabels = [pretty_names.get(label.get_text(), label.get_text())
               for label in ax.get_yticklabels()]
ax.set_yticklabels(yticklabels, fontsize=20, weight='bold')
ax.set_xticklabels(ax.get_xticklabels(),fontsize=18, weight='bold')
# Label and title
ax.set_xlabel("SHAP value", fontsize=23, labelpad=18, weight='bold')
ax.set_ylabel("")  # optional: hide redundant label
ax.set_ylim(-1, len(yticklabels) - 1 + 0.8)

# Tweak layout
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "SHAP1.0.png"), dpi=300)
plt.show()

In [ ]:
# ==== Save the model as a pipeline ====
class LabelEncoderWrapper(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.encoders = {}

    def fit(self, X, y=None):
        for col in X.columns:
            le = LabelEncoder()
            le.fit(X[col])
            self.encoders[col] = le
        return self

    def transform(self, X):
        X_encoded = X.copy()
        for col in X.columns:
            X_encoded[col] = self.encoders[col].transform(X[col])
        return X_encoded

    def inverse_transform(self, X):
        X_decoded = X.copy()
        for col in X.columns:
            X_decoded[col] = self.encoders[col].inverse_transform(X[col])
        return X_decoded

cat_col = ['SID']  # species id
target_col = 'CR'  # assuming target column name is 'CR'
numeric_cols = ['H(ft)', 'DBH(inch)', 'CD(%)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'Lat', 'Long', 'CR_pred']
feature_cols = cat_col + numeric_cols

# Prepare X, y, species
X = df[feature_cols]
y = df[target_col]
species = df[cat_col].astype(str)

# Train/val split stratified by species
X_train, X_val, y_train, y_val, species_train, species_val = train_test_split(
    X, y, species, test_size=0.2, random_state=SEED, stratify=species
)

# encoded species code
le = LabelEncoderWrapper()
scaler = StandardScaler()
preprocessor = ColumnTransformer(
    transformers=[
         ('num', scaler, numeric_cols),
        ('cat', le, cat_col)
    ]
)

preprocessor.fit(X_train)

pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('regressor', model)
])

pipeline_name = f'{try_num}-XGBoostGlobal-pipeline.pkl'
joblib.dump(pipeline, os.path.join(model_dir, pipeline_name))

# Prediction & Mapping

In [ ]:
# ==== NFI 불러오기 및 변수 설정 ====
nfi_file = os.path.join(data_dir, r"NFI6-7-Immok-Filtered.xlsx")
try:
    nfi_data = pd.read_excel(nfi_file, sheet_name = "NFI6-7")
except:
    print("Sheet name 'NFI6-7' need to exist in the NFI file.")

# 1회만 시행
nfi_data['수고'] = nfi_data['수고'] / 100 # m로 변환
nfi_data['평균수관밀도(%)'] = nfi_data['평균수관밀도(%)'] / 100 # decimal 로 변환

# SEED & 샘플 크기 설정
SEED = 400
SAMPLE_SIZE = 1000000

In [ ]:
#  ==== 평균수관밀도(%) ('DMCLS_CD') 분포 추정 ====
cd_dist = buildDistribution()
attribute = '평균수관밀도(%)'
lower = 0.
upper = 1.0
cd_best_fit, cd_results = cd_dist.find_best_fit_distribution(nfi_data, lower, upper, attribute)
cd_samples = cd_dist.create_samples(attribute_name=attribute, min_bound=lower, max_bound=upper, sample_size=SAMPLE_SIZE, SEED=SEED)
cd_dist.plot_distribution()

In [ ]:
# ==== 수고(m) (HEIGHT)의 분포 추정 ====
h_dist = buildDistribution()
attribute = '수고'
lower = 0
upper = 40
h_best_fit, h_results = h_dist.find_best_fit_distribution(nfi_data, lower, upper, attribute)
h_samples = h_dist.create_samples(attribute_name=attribute, min_bound=lower, max_bound=upper, sample_size=SAMPLE_SIZE, SEED=SEED)
h_dist.plot_distribution()

In [ ]:
# ==== DBH(inch) ('DNST_CD')의 분포 추정 ====
dbh_dist = buildDistribution()
attribute = '흉고직경'
lower = 0
upper = 110
dbh_best_fit, dbh_results = dbh_dist.find_best_fit_distribution(nfi_data, lower, upper, attribute)
dbh_samples = dbh_dist.create_samples(attribute_name=attribute, min_bound = lower, max_bound = upper, sample_size=SAMPLE_SIZE, SEED=1000)
dbh_dist.plot_distribution()

In [ ]:
# ==== Read & Preprocess 임상도 ====
# gdb에 레이어가 존재한다면 첫번째 레이어 읽어오기
if fiona.listlayers(gdb_dir):
    layer = fiona.listlayers(gdb_dir)
    imsang = gpd.read_file(os.path.join(gdb_dir), layer=layer[0])

# DBH, 수고, 수관비율 값만 저장
imsang[['DBH(cm)', 'Height(m)', 'CD(%)']] = np.zeros((len(imsang), 3))


# ==== 1번만 실행 ====
imsang.reset_index(names='ID', inplace=True)
imsang['ID'] = imsang['ID'] + 1
no_imsang_code = [str(i) for i in [78, 81, 82, 83, 91, 92, 93, 94, 95, 99]] # 산림지만 추출하기
filtered_imsang = imsang[~imsang['KOFTR_GROU'].isin(no_imsang_code)]

print("추출한 임상도 크기: ", len(filtered_imsang.index), len(imsang.index))
# HEIGHT 코드 정돈
imsang.loc[:, 'HEIGHT'] = imsang['HEIGHT'].apply(lambda x: '40' if x == '42' else x)
imsang.loc[:, 'HEIGHT'] = imsang['HEIGHT'].apply(lambda x: '00' if x == '0' else x)
imsang.loc[:, 'HEIGHT'] = imsang['HEIGHT'].apply(lambda x: '16' if x == '15' else x)

In [ ]:
# ==== Sampling 실행 ====
raster_file = os.path.join(raster_dir, "imsang_Raster2.tif")
si = GPUSamplingImsang(imsang, imsang_raster = raster_file,
                       h_samples=h_samples, dbh_samples=dbh_samples, cd_samples=cd_samples)
si.run_sampling(result_dir=raster_result_dir, patch_size=256)

In [ ]:
# ==== Allometric model 파라미터 불러오기 및 수종 대체 ====
params_file = os.path.join(model_dir, r"HM변형1.0-params.csv")
substitute_file = os.path.join(data_dir, r"Alternative_Species_Model.csv")
allo_params = np.loadtxt(params_file, dtype='object', encoding='utf-8', skiprows=1)
substitute_info = np.loadtxt(substitute_file, dtype='object', encoding='utf-8', skiprows=1, usecols=0)
allo_params = np.array([row.split(',') for row in allo_params])
substitute_info = np.array([row.split(',')[:4] for row in substitute_info])
substitute_info = substitute_info[:-10, [0, 2]].astype('int')

# 대체수종으로 바꾸기
target_species = allo_params[allo_params[:, 2] == '', 0].astype('int')
substitute_species = [row[:2] for row in substitute_info if len(row) > 1 and row[0] in target_species]
substitute_dict = {target : subs for target, subs in substitute_species}
allo_params = allo_params[:,  [0, 2, 3, 4, 5]]
for key, value in substitute_dict.items():
    allo_params[allo_params[:, 0] == str(key), 1:] = allo_params[allo_params[:, 0] == str(value), 1:]
allo_params = np.where(allo_params != '', allo_params, '0')
allo_params = allo_params.astype('float')
# 최종 파라미터 dictionary: {수종코드 : list(파라미터(4))}
allo_params_dict = {row[0] : row[1:] for row in allo_params}
allo_keys_np = np.fromiter(allo_params_dict.keys(), dtype=np.int64) # fromiter: convert python iterable to the array
# shape -> (n_species, 4)
allo_vals_np = np.vstack([np.concatenate([np.asarray(allo_params_dict[k], dtype=np.float64), np.full((4,), fill_value=-999.)]) for k in allo_keys_np])
allo_vals_np
# sort for searchsorted
allo_order = np.argsort(allo_keys_np)
allo_keys_sorted = cp.asarray(allo_keys_np[allo_order], dtype=cp.int64)
allo_vals_sorted = cp.asarray(allo_vals_np[allo_order], dtype=cp.float64)  # columns: b1,b2,b3,c
allo_vals_sorted[-1, :] = cp.asarray(np.concatenate([allo_params_dict[11], allo_params_dict[32]]))
# target_species = [19, 20, 21, 43, 45, 65, 66, 67, 68, 77] # XGBoost

# 대체수종
map_keys = cp.asarray(cp.array([19, 20, 21, 60, 63, 65, 67], dtype=cp.int64))
map_vals = cp.asarray(cp.array([11, 11, 32, 32, 32, 32, 32], dtype=cp.int64))

In [ ]:
# ==== input 데이터 불러오기 ====
block_size = 512

rasters = glob(os.path.join(raster_dir, '*.tif'))
dem_file = next((f for f in rasters if 'DEM' in f), None)
slope_file = next((f for f in rasters if 'DEM' in f), None)
aspect_file = next((f for f in rasters if 'Aspect' in f), None)
height_file = next((f for f in rasters if 'HEIGHT' in f), None)
dbh_file = next((f for f in rasters if 'DMCLS_CD' in f), None)
cd_file = next((f for f in rasters if 'DNST_CD' in f), None)
species_file = next((f for f in rasters if 'KOFTR_INT' in f), None)

raster_paths = {
    'dem' : dem_file, 'slope' : slope_file, 'aspect' : aspect_file,
    'height' : height_file, 'dbh' : dbh_file, 'density' : cd_file,
    'species' : species_file
    }

for rp in raster_paths.items():
    if rp[1] == None:
        raise FileNotFoundError(f"Raster of {rp[0]} doesn't exist in '{raster_dir}'")

In [ ]:
# ==== Pipeline 불러오기 ====
class LabelEncoderWrapper(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.encoders = {}

    def fit(self, X, y=None):
        for col in X.columns:
            le = LabelEncoder()
            le.fit(X[col])
            self.encoders[col] = le
        return self

    def transform(self, X):
        X_encoded = X.copy()
        for col in X.columns:
            X_encoded[col] = self.encoders[col].transform(X[col])
        return X_encoded

    def inverse_transform(self, X):
        X_decoded = X.copy()
        for col in X.columns:
            X_decoded[col] = self.encoders[col].inverse_transform(X[col])
        return X_decoded
        
# pipeline_name = 'HM변형1.0-XGBoostGlobal-pipeline.pkl'
ml_model = joblib.load(os.path.join(model_dir, pipeline_name))

# 모델 유효성 검사
print(check_model_validity(ml_model))
ml_model.named_steps.values()

In [ ]:
# ===== Model Prediction =====
# Global variables
STOP_ON_ERROR = True
initialize_log = True
if initialize_log:
    write_log("START PREDICTION", "Hybrid_model", block_num=None, initialize=initialize_log)

# Prediction
with rasterio.open(species_file) as species_ras, \
     rasterio.open(dem_file) as dem_ras, \
     rasterio.open(slope_file) as slp_ras, \
     rasterio.open(aspect_file) as asp_ras, \
     rasterio.open(height_file) as h_ras, \
     rasterio.open(dbh_file) as dbh_ras, \
     rasterio.open(cd_file) as cd_ras:
         
    ref_height, ref_width = dem_ras.height, dem_ras.width
    ref_transform = dem_ras.transform
    profile = dem_ras.profile.copy()
    b_height, b_width = profile.get('blockysize', block_size), profile.get('blockxsize', block_size)
    block_cnt = int(np.ceil(ref_height / b_height)) * int(np.ceil(ref_width / b_width))
    dem_nodata = dem_ras.nodata if dem_ras.nodata else -9999.
    species_nodata = species_ras.nodata if species_ras.nodata else -9999.
    h_nodata = h_ras.nodata if h_ras.nodata else -9999.
    dbh_nodata = dbh_ras.nodata if dbh_ras.nodata else -9999.
    cd_nodata = cd_ras.nodata if cd_ras.nodata else -9999.

    # Check: all rasters must align
    for src in [species_ras, slp_ras, asp_ras, h_ras, dbh_ras, cd_ras]:
        assert src.crs == dem_ras.crs
        assert src.transform == dem_ras.transform
        assert (src.width, src.height) == (dem_ras.width, dem_ras.height)

    # update profile to process the large data
    profile.update(driver = 'GTiff', dtype = 'float32', count=1, nodata=dem_nodata,
                  tiled=True, blockxsize=b_width, blockysize=b_height,
                  compress = 'ZSTD', predictor=3, bigtiff='YES')

    dst_files = {
        attr : rasterio.open(os.path.join(raster_dir, f"{attr}4.tif"), 'w', **profile)
        for attr in ['CR', 'CBH']
    }
    print("All the files are ready!")
    # ===== 블록 단위 처리 =====
    # block 단위로 raster 불러오기: height, dbh, cd, elev, slope, azimuth, id(species)
    try:
        # iterate by windows; remove [:100 ] & list() to do full
        for ji, win in tqdm(list(species_ras.block_windows(1))[1820:], desc="Creating CBH...", total = block_cnt):
            try:
                # ===== read blocks to GPU =====
                species_block = cp.asarray(species_ras.read(1, window=win))
                dem_block = cp.asarray(dem_ras.read(1, window=win))
                slp_block = cp.asarray(slp_ras.read(1, window=win))
                asp_block = cp.asarray(asp_ras.read(1, window=win))
                h_block = cp.asarray(h_ras.read(1, window=win))
                dbh_block = cp.asarray(dbh_ras.read(1, window=win))
                cd_block = cp.asarray(cd_ras.read(1, window=win))
                
                # allocate outputs (GPU)
                out_dtype = cp.float32
                out_cr = cp.full(dem_block.shape, dem_nodata, dtype=out_dtype)
                out_cbh = cp.full(dem_block.shape, dem_nodata, dtype=out_dtype)
                
                # Validity check & create mask
                # species_ok  = create_valid_mask(id_block,  species_nodata)
                dem_ok = create_valid_mask(dem_block, dem_nodata)
                h_ok   = create_valid_mask(h_block,   h_nodata)
                # dbh_ok = create_valid_mask(dbh_block, dbh_nodata)
                # cd_ok  = create_valid_mask(cd_block,  cd_nodata)
                
                valid_mask = (dem_ok & h_ok)
                if not bool(valid_mask.any()):
                    dst_files['CR'].write(out_cr.get(), 1, window=win)
                    dst_files['CBH'].write(out_cbh.get(),1, window=win)
                    write_log(f"No valid pixel exisats in BLOCK{ji}.","Hybrid_model",sum(ji))
                    continue
                    
                # ===== unit conversion (only for valid pixels) =====
                # why only for valid pixels? To prevent the transformation of no data
                # height, dbh, cd: 단위변환
                h_ft_block = cp.where(valid_mask, h_block * m_to_ft, h_block)
                dbh_block = cp.where(valid_mask, dbh_block * cm_to_inch, dbh_block)
                # Activate if you need it: Elevation (m -> hectometer), Slope (deg -> tan), Aspect (deg -> rad)
                dem_block = cp.where(valid_mask, dem_block / 100.0, dem_block)
                slp_block = cp.where(valid_mask, cp.tan(cp.deg2rad(slp_block)), slp_block)
                asp_block = cp.where(valid_mask, cp.deg2rad(asp_block), asp_block)
                
                # ===== Lat/Long per pixel (CPU -> GPU) =====
                h, w = species_block.shape # need to use actual block size!
                rows, cols = np.meshgrid(np.arange(h), np.arange(w), indexing='ij') # row, col index번호로만 채운 array 각각 반환
                aff = transform(win, ref_transform) # window에 해당되는 변환정보 반환
                x, y = xy(aff, rows.ravel(), cols.ravel(), offset='center')
                xs = np.asarray(x).reshape(h, w)
                ys = np.asarray(y).reshape(h, w)
                long_cp = cp.asarray(xs)
                lat_cp = cp.asarray(ys)
    
                
                # ===== Create species code block (Deleted) ===== 
                # Instead, rasterize the 임상도 with "KOFTR_GROU" using ArcGIS
                # searchsorted 함수로 'key'를 'index'로 변환하여 array에서 작업할 수 있음.
                """
                keys = cp.asarray(np.fromiter(index_species_dict.keys(), dtype=np.int64))
                vals = cp.asarray(np.fromiter(index_species_dict.values(), dtype=np.int64))
                order = cp.argsort(keys) # 오름차순으로 정렬
                keys_s = keys[order]
                vals_s = vals[order]
                flat_ids = species_block.ravel().astype(keys_s.dtype, copy=False) # imsang id 저장된 raster flatten
                pos = cp.searchsorted(keys_s, flat_ids) # raster id 값이 key array에서 위치하는 index 값 반환
                match = (pos < keys_s.size) & (keys_s[pos] == flat_ids) # pos가 key array의 길이를 초과하지 않고, 해당 pos에서의 key값이 raster id의 id 값과 동일한지 확인
                out = cp.full(flat_ids.size, dem_nodata, dtype=cp.int64)
                out[pos[match]] = vals_s[pos[match]] # 일치하는 position에서 value값 반환
                species_block = out.reshape(species_block.shape)
                """
    
                # ===== Model Prediction per pixel (vectorized) =====
                # 대체수종으로 코드 대체
                sid_flat = species_block.ravel().astype(cp.int64, copy=False) # (N, 1)
                p = cp.searchsorted(map_keys, sid_flat)
                m = (p < map_keys.size) & (map_keys[p] == sid_flat)
                sid_flat_mapped = sid_flat.copy()
                sid_flat_mapped[m] = map_vals[p[m]]
                # only get the pixels with a valid value
                v_flat = valid_mask.ravel() # (I, J)
                pos2 = cp.searchsorted(allo_keys_sorted, sid_flat_mapped) # (K, )
                match2 = (pos2 < allo_keys_sorted.size) & (allo_keys_sorted[pos2] == sid_flat_mapped) # (K1, )
                pred_mask = v_flat & match2 # no data가 아니면서, 파라미터가 존재하는 수종코드인 pixel만 선택
    
                # ===== Count the valid pixels =====
                try:
                    valid_cnt = int(valid_mask.sum().get())
                    pred_cnt = int(pred_mask.sum().get())
                except Exception:
                    # CPU arr로 반환되는 겨웅
                    valid_cnt = int(valid_mask.sum())
                    pred_cnt = int(pred_mask.sum())
                write_log(f"[block {ji}] valid_mask={valid_cnt}, pred_mask={pred_cnt}", "Hybrid_model", sum(ji))
                    
                if not bool(pred_mask.any()):
                    # any species don't have parameters -> wirte nodata and continue
                    dst_files['CR'].write(out_cr.get(), 1, window=win)
                    dst_files['CBH'].write(out_cbh.get(), 1, window=win)
                    write_log(f"Block{ji}: no matching allomeric parameters for any pixels", "Hybrid_model", sum(ji))
                    continue

                # ===== predict using allometric =====
                params = cp.take(allo_vals_sorted, pos2[pred_mask], axis=0) # (K2, 4)
                params_np = params.get()
    
                idx = cp.nonzero(pred_mask)[0] # (K2, ): K2 <= K1
                h_ft_vec = h_ft_block.ravel()[idx]
                h_m_vec = h_block.ravel()[idx]
                dbh_vec = dbh_block.ravel()[idx]
                cd_vec = cd_block.ravel()[idx]
                dem_vec = dem_block.ravel()[idx]
                slp_vec = slp_block.ravel()[idx]
                asp_vec = asp_block.ravel()[idx]
                lat_vec = lat_cp.ravel()[idx]
                long_vec = long_cp.ravel()[idx]
                sid_vec = sid_flat_mapped[idx]
    
                # create dataframe: ['H(ft)', 'DBH(inch)', 'CD(%)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'SID_ENC', 'Lat', 'Long', 'CR_pred']
                df_input = pd.DataFrame({
                            'H(ft)'       : h_ft_vec.get(),
                            'DBH(inch)'   : dbh_vec.get(),
                            'CD(%)'       : cd_vec.get(),
                            'Elev(hm)'    : dem_vec.get(),
                            'Slope(tan)'  : slp_vec.get(),
                            'Azimuth(rad)': asp_vec.get(),
                            'SID'         : sid_vec.get().astype(np.int64),
                            'Lat'         : lat_vec.get(),
                            'Long'        : long_vec.get(),
                        })
                # Check: length compliance
                assert len(params_np) == len(df_input),\
                f"len(params_np)={len(params_np)} vs len(df_input)={len(df_input)} (idx={len(idx)}, pred_mask_true={int(pred_mask.sum())})"
                
                # ===== predict with hybrid model =====
                cr_pred = hybrid_model(ml_model, params_np, df_input)
            
                # ===== Save the result =====
                out_cr.ravel()[idx] = cp.asarray(cr_pred, dtype=out_dtype)
                cbh_pred = cp.asarray(cr_pred) * h_m_vec
                out_cbh.ravel()[idx] = cp.asarray(cbh_pred, dtype=out_dtype)
    
                dst_files['CR'].write(out_cr.get(), 1, window=win)
                dst_files['CBH'].write(out_cbh.get(), 1, window=win)
                
            # =====Exception: write nodata and continue =====
            except Exception as e:
                write_log(f"BLOCK {ji} failed: {type(e).__name__}: {e}", "Prediction", sum(ji))
                traceback.print_exc()
                if STOP_ON_ERROR: raise
                else:
                    try:
                        dst_files['CR'].write(out_cr.get(), 1, window=win)
                        dst_files['CBH'].write(out_cbh.get(), 1, window=win)
                    except Exception:
                        pass
                    continue
            
    # ===== Finally: Always Run this code despite excpetions =====
    finally:
        for f in dst_files.values():
            f.close()